In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl

from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh, FastMidpointMapper
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem, enforce_dirichlet_boundary
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre

In [ ]:
n_refinements = 1
p0 = 2
knots1 = np.array([0,0,0, 0.5,0.5, 1,1,1], dtype=np.float64)
#knots1 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 = np.array([0,0,0, 0.5,1,1,1], dtype=np.float64)
#knots2 = np.array([-1, -1, -1, 0,0,1,1,1], dtype=np.float64)
knots2 = refine(knots2, p0, n_times=n_refinements-1)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=2)
    # hs_dual.refine(cells, level, refine_neighbours=False, buffer_zone_size=max(p0, p0_dual))
pass
hs.hmesh.plot_cells()

In [ ]:
def map_uv_to_xy_turn(uv_points, nodes_per_cell=4):
    """Maps parametric [0,1]^2 to a 4-cell L-shape without polar singularities."""
    # Ensure points are structured as complete cells
    assert len(uv_points) % nodes_per_cell == 0, "uv_points must be grouped by cells."
    
    # Reshape points to (N_cells, 4, 2)
    uv_cells = uv_points.reshape(-1, nodes_per_cell, 2)
    xy_cells = np.zeros_like(uv_cells)
    
    def bilinear(u_loc, v_loc, C00, C10, C01, C11):
        """Standard isoparametric Q1 interpolation"""
        return ((1-u_loc)*(1-v_loc)*C00 + 
                u_loc*(1-v_loc)*C10 + 
                (1-u_loc)*v_loc*C01 + 
                u_loc*v_loc*C11)
    
    # Map cell by cell using the cell center to uniquely identify the block
    for i in range(len(uv_cells)):
        u = uv_cells[i, :, 0]
        v = uv_cells[i, :, 1]
        
        # Cell centers bypass boundary ambiguities
        u_c = np.mean(u)
        v_c = np.mean(v)
        
        if u_c < 0.25:
            if v_c >= 0.5:
                # Cell 0 -> P0 (Bottom-Left)
                C00, C10, C01, C11 = [-0.5, -1], [-0.5, -0.25], [-1, -1], [-1, 0]
                u_loc, v_loc = u * 4.0, (v-0.5) * 2.0
            else:
                # Cell 2 -> P1 
                C00, C10, C01, C11 = [0, -1], [0, -0.5], [-0.5, -1], [-0.5, -0.25]
                u_loc, v_loc = u * 4.0, v * 2.0
                
                
        elif u_c < 0.5:
            if v_c >= 0.5:
                # Cell 1 -> P2
                C00, C10, C01, C11 = [-0.5, -0.25], [-0.5, 0.5], [-1, 0], [-1, 1]
                u_loc, v_loc = (u-0.25) * 4.0, (v - 0.5) * 2.0
            else:
                # Cell 3 -> P3 
                C00, C10, C01, C11 = [0, -0.5], [0, 0], [-0.5, -0.25], [-0.5, 0.5]
                u_loc, v_loc = (u - 0.25) * 4.0, v * 2.0
                
        elif u_c < 0.75:
            if v_c < 0.5:
                # Cell 4 -> P4 (Bottom-Mid-Right) - Connects to P3's top edge!
                C00, C10, C01, C11 = [0, 0], [0.5, 0], [-0.5, 0.5], [0.25, 0.5]
                u_loc, v_loc = (u - 0.5) * 4.0, v * 2.0
            else:
                # Cell 5 -> P5 (Top-Mid-Right)
                C00, C10, C01, C11 = [-0.5, 0.5], [0.25, 0.5], [-1, 1], [0, 1]
                u_loc, v_loc = (u - 0.5) * 4.0, (v - 0.5) * 2.0
                
        else:
            if v_c < 0.5:
                # Cell 6 -> P6 (Bottom-Right)
                C00, C10, C01, C11 = [0.5, 0], [1, 0], [0.25, 0.5], [1, 0.5]
                u_loc, v_loc = (u - 0.75) * 4.0, v * 2.0
            else:
                # Cell 7 -> P7 (Top-Right)
                C00, C10, C01, C11 = [0.25, 0.5], [1, 0.5], [0, 1], [1, 1]
                u_loc, v_loc = (u - 0.75) * 4.0, (v - 0.5) * 2.0
                
        C00, C10 = np.array(C00), np.array(C10)
        C01, C11 = np.array(C01), np.array(C11)
        
        xy = bilinear(u_loc[:, None], v_loc[:, None], C00, C10, C01, C11)
        xy_cells[i] = xy
        
    return xy_cells.reshape(-1, 2)

disconnected_mesh, thb_operators, N_max, physical_cells_midpoints = build_mesh(hs=hs, mapping=map_uv_to_xy_turn)

In [ ]:
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

plotter = pyvista.Plotter()
plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
plotter.view_xy()
plotter.show(jupyter_backend="static")
print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
def outer_boundary(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], 1.) & (x[1]>=-1e-10)
    on_top = np.isclose(x[1], 1.) & (x[0]<=1.)
    return on_left|on_bottom|on_right|on_top

legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, outer_boundary)
facet_values = np.full_like(boundary_facets, 1, dtype=np.int32)
boundary_tags = dolfinx.mesh.meshtags(
    disconnected_mesh, 
    facet_dim,
    boundary_facets,
    facet_values
)
custom_metadata = {"quadrature_degree": 16}
ds = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=boundary_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
a0 = ufl.inner(ufl.grad(u), ufl.grad(v)) * dx_custom

my_x = ufl.SpatialCoordinate(disconnected_mesh)
n_normal = ufl.FacetNormal(disconnected_mesh)
r_radius = ufl.sqrt(my_x[0]**2+my_x[1]**2+1e-15)
theta_angle = ufl.atan2(my_x[1], my_x[0])
# restrict theta to [0, 2\pi)
theta_angle = ufl.conditional(condition=ufl.lt(left= theta_angle,right= 0.), true_value= theta_angle+ 2*np.pi, false_value= theta_angle)
#u_bar = dolfinx.fem.FunctionSpace(V)
u_bar = r_radius**(2./3.)*ufl.sin(2./3. * theta_angle)
g = ufl.dot(ufl.grad(u_bar), n_normal)
# ds(1) because we defined our subdomain_data to have label "1" at the boundaries of interest.
L_neumann = ufl.inner(g, v)*ds(1)

msh = disconnected_mesh

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=True)
custom_mapping = FastMidpointMapper(hs, physical_cells_midpoints)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh, N_max = N_max, 
                                      thb_operators=thb_operators, mapping_function=custom_mapping)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=1)
local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(disconnected_mesh, a0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L = make_linear_kernel(disconnected_mesh, L_neumann, padded_dofs=N_max, local_dofs=local_dofs)

In [ ]:
facet_dim = msh.topology.dim-1
# fenicsx numbers cell facets (=edges) internally, as
# 0 for bottom, 1 for left, 2 for right and 3 for top 
# since all cells are disconnected, there should be in total 
# 4*n_cells boundary facets.
boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, outer_boundary)

In [ ]:
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_neumann = {dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L.address, boundary_entities, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_neumann, #give the adress of stuff to integrate
        coefficients=[C_func._cpp_object], # holds C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_on_inner_corner_turn(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        current_dirichlet_indices = np.isclose(basis[:, 0, :-1], np.zeros((hs.degrees[0]+1)))
        current_dirichlet_indices = np.all(current_dirichlet_indices, axis=-1)
        current_dirichlet_indices = np.nonzero(current_dirichlet_indices)[0]
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  current_dirichlet_indices,
                                                  assume_unique=True)
    pass

    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

In [ ]:
#forbidden_indices = get_spline_indices_on_inner_corner_turn(hs, dofmap)


In [ ]:
forbidden_indices = enforce_dirichlet_boundary(hs, dofmap, bottom=True)
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=forbidden_indices,
                         dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline,
                         iterative=False, return_A=True)

In [ ]:
# Extract the CSR (Compressed Sparse Row) arrays from PETSc
indptr, indices, data = A.getValuesCSR()

# Get the global size of the matrix
shape = A.getSize()

# Create a SciPy CSR matrix
A_scipy = sp.csr_array((data, indices, indptr), shape=shape)

# print(f"Matrix shape: {A_scipy.shape}")
# print(f"Number of non-zeros: {A_scipy.nnz}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(7, 7))
# # plt.spy plots the non-zero entries of a matrix
# plt.spy(A_scipy, markersize=2, color='lightseagreen')
# plt.title("Sparsity Pattern of THB-Spline mass Matrix")
# plt.show()

In [ ]:
# Map the global B-spline coefficients back to local Legendre coefficients
# u_dg = dolfinx.fem.Function(V)
# c_values = C_func.x.array.reshape((-1, N_max, (hs.degrees[0]+1)**2))
# # x_vec is the solution vector
# for local_idx in range(msh.topology.index_map(msh.topology.dim).size_local):
#     spline_dofs = padded_cells_to_dofs[local_idx]
#     u_spline_local = x_vec[spline_dofs]
    
#     G = c_values[local_idx, :, :]
#     u_dg_local = G.T @ u_spline_local
    
#     dg_dofs = V.dofmap.cell_dofs(local_idx)
#     u_dg.x.array[dg_dofs] = u_dg_local
u_dg = map_spline_to_legendre(hs, V, C_func, N_max, disconnected_mesh, padded_cells_to_dofs, x_vec)
u_dg.x.scatter_forward()


# Define the exact analytic solution using UFL
my_x = ufl.SpatialCoordinate(disconnected_mesh)
r_radius = ufl.sqrt(my_x[0]**2 + my_x[1]**2 + 1e-15) # 1e-15 prevents singularity at origin
theta_angle = ufl.atan2(my_x[1], my_x[0])

# Restrict theta to [0, 2\pi)
theta_angle = ufl.conditional(
    ufl.lt(theta_angle, 0.0), 
    theta_angle + 2*np.pi, 
    theta_angle
)

# Exact solution u_bar
u_bar = (r_radius**(2.0/3.0)) * ufl.sin((2.0/3.0) * theta_angle)

# Compute L2 Error: sqrt( \int (u_bar - u_dg)^2 dx )
error_L2_form = dolfinx.fem.form(ufl.inner(u_bar - u_dg, u_bar - u_dg) * dx_custom)
error_L2_sq = dolfinx.fem.assemble_scalar(error_L2_form)
l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_L2_sq, op=MPI.SUM))

# Compute H1 Semi-norm (Gradient) Error: sqrt( \int |grad(u_bar) - grad(u_dg)|^2 dx )
# This is the "energy" error and is crucial for elliptic PDEs!
error_H1_form = dolfinx.fem.form(ufl.inner(ufl.grad(u_bar) - ufl.grad(u_dg), 
                                           ufl.grad(u_bar) - ufl.grad(u_dg)) * dx_custom)
error_H1_sq = dolfinx.fem.assemble_scalar(error_H1_form)
h1_error = np.sqrt(disconnected_mesh.comm.allreduce(error_H1_sq, op=MPI.SUM))

# Compute Exact Norms (for Relative Error calculations)
norm_L2_form = dolfinx.fem.form(ufl.inner(u_bar, u_bar) * ufl.dx)
exact_L2_norm = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(norm_L2_form), op=MPI.SUM))

norm_H1_form = dolfinx.fem.form(ufl.inner(ufl.grad(u_bar), ufl.grad(u_bar)) * ufl.dx)
exact_H1_norm = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(norm_H1_form), op=MPI.SUM))

# Print results
print(f"Absolute L2 Error: {l2_error:.2e}")
print(f"Relative L2 Error: {l2_error / exact_L2_norm:.2e}\n")

print(f"Absolute H1 Error: {h1_error:.3e}")
rel_err = h1_error / exact_H1_norm
print(f"Relative H1 Error: {rel_err:.3e}")
#print(f"dofs = {A_scipy.shape[0]-1}")

In [ ]:
# Create a DG0 space (one value per cell)
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)

hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array

# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(hQ**2*ufl.inner(ufl.grad(u_bar - u_dg), ufl.grad(u_bar - u_dg)) * v * dx_custom)
err_cells = dorfler_marking(hierarchical_space=hs, theta=0.5, 
                            local_error_form=local_error_form)

In [ ]:
import dolfinx.plot
import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
v_plot_elt = basix.ufl.element(
    "DG", 
    "quadrilateral", 
    degree=p0+2
)
V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
u_plot = dolfinx.fem.Function(V_plot)
error_ufl = ufl.sqrt((u_dg-u_bar)**2)
#error_ufl = u_bar
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)
#u_plot.interpolate(u_dg) #How to plot |u_dg-u_bar|? 

# 3. Now use V_plot for the VTK mesh generation
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
# grid.point_data["u"] = u_error.x.array.real
# grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
#plotter = pyvista.Plotter(window_size=[2500, 2500])
#grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
#plotter.add_mesh(grid_shrink, show_edges=False, cmap="jet", show_scalar_bar=False)
#plotter.view_xy()
#plotter.show(jupyter_backend="static")
#plotter.screenshot("l_shape_error6.png", transparent_background=threshold_value)
#plotter.show()
# print(f"{u_error.x.array.min():.5e}")